# Notebook 02c — Generate `channel_data_normalized.pkl` for SMAP (Kaggle-ready)

Companion to `02_preprocessing.ipynb`, adapted to run standalone on Kaggle with **no path
editing required** — it auto-discovers your SMAP raw files anywhere under `/kaggle/input`
via `os.walk`, same pattern as the SWaT/WADI notebooks' `_find()` helper.

### ▶ Run instructions
1. Attach your SMAP raw data as a Kaggle Dataset input — the `labeled_anomalies.csv` file
   and the `train/`/`test/` folders full of per-channel `.npy` files, in any subfolder
   structure (this notebook finds them regardless of nesting).
2. **Save Version → "Save & Run All (Commit)"**. This is CPU-only and fast (a few minutes) —
   no GPU needed for this notebook.
3. Attach this notebook's own output (containing `channel_data_normalized.pkl` and
   `config.json`) as an additional input to `06_smap_maml_clean.ipynb`, alongside
   `task_splits_25feat.json` (already in the `MAML_AE` repo — add it to the same or a
   separate input dataset).


In [1]:
import os, json, pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)
OUTPUT_PATH = "/kaggle/working"
os.makedirs(OUTPUT_PATH, exist_ok=True)

def find_file(name):
    for root, _, files in os.walk("/kaggle/input"):
        if name in files:
            return os.path.join(root, name)
    return None

def find_dir(name):
    for root, dirs, _ in os.walk("/kaggle/input"):
        if name in dirs:
            return os.path.join(root, name)
    return None

LABEL_PATH = find_file("labeled_anomalies.csv")
TRAIN_DIR = find_dir("train")
TEST_DIR = find_dir("test")
print("labeled_anomalies.csv:", LABEL_PATH)
print("train dir:", TRAIN_DIR)
print("test dir:", TEST_DIR)
assert LABEL_PATH and TRAIN_DIR and TEST_DIR, "could not locate SMAP raw files under /kaggle/input"


labeled_anomalies.csv: /kaggle/input/smap-nasa/labeled_anomalies.csv
train dir: /kaggle/input/smap-nasa/data/data/train
test dir: /kaggle/input/smap-nasa/data/data/test


## Build windows exactly as `02_preprocessing.ipynb` does

Window size 30, anomaly-window stride 5 (test data, from labeled anomaly sequences),
normal-window stride 10 (train data, capped at 500 windows/channel), MinMax-scaled with
the scaler fit on that channel's *training* data only.

In [2]:
labels_df = pd.read_csv(LABEL_PATH)
smap_channels = labels_df[labels_df['spacecraft']=='SMAP']['chan_id'].tolist()
msl_channels = labels_df[labels_df['spacecraft']=='MSL']['chan_id'].tolist()
all_channels = smap_channels + msl_channels
print(f"total channels: {len(all_channels)}")

def create_windows(data, window_size=30, stride=1):
    windows = []
    for start in range(0, len(data) - window_size + 1, stride):
        windows.append(data[start:start+window_size])
    return np.array(windows)

WINDOW_SIZE = 30
STRIDE_ANOMALY = 5
STRIDE_NORMAL = 10
MAX_NORMAL = 500

channel_data_normalized = {}
problem_channels = []

for ch in all_channels:
    try:
        train_raw = np.load(f"{TRAIN_DIR}/{ch}.npy")
        test_raw = np.load(f"{TEST_DIR}/{ch}.npy")
        scaler = MinMaxScaler()
        scaler.fit(train_raw)
        train_norm = np.clip(scaler.transform(train_raw), 0, 1)
        test_norm = np.clip(scaler.transform(test_raw), 0, 1)

        ch_info = labels_df[labels_df['chan_id'] == ch]
        anomaly_seqs = eval(ch_info['anomaly_sequences'].values[0])

        anomaly_windows = []
        for seq in anomaly_seqs:
            start, end = int(seq[0]), int(seq[1])
            segment = test_norm[max(0, start):min(end + WINDOW_SIZE, len(test_norm))]
            if len(segment) >= WINDOW_SIZE:
                anomaly_windows.extend(create_windows(segment, WINDOW_SIZE, STRIDE_ANOMALY))

        if len(anomaly_windows) == 0:
            problem_channels.append(ch)
            continue
        anomaly_windows = np.array(anomaly_windows)

        all_normal = create_windows(train_norm, WINDOW_SIZE, STRIDE_NORMAL)
        if len(all_normal) > MAX_NORMAL:
            idx = np.random.choice(len(all_normal), MAX_NORMAL, replace=False)
            all_normal = all_normal[idx]

        channel_data_normalized[ch] = {
            'anomaly_windows': anomaly_windows,
            'normal_windows': all_normal,
            'n_anomaly': len(anomaly_windows),
            'n_normal': len(all_normal),
            'spacecraft': 'SMAP' if ch in smap_channels else 'MSL',
        }
    except Exception as e:
        problem_channels.append(ch)
        print(f"problem with {ch}: {e}")

print(f"channels normalized: {len(channel_data_normalized)}  problems: {len(problem_channels)} {problem_channels}")


total channels: 82


channels normalized: 81  problems: 0 []


## Save

Saves `channel_data_normalized.pkl` and a `config.json` record of the settings used.
`task_splits_25feat.json` is **not** regenerated here — use the one already in the repo
(its exact channel-to-split assignment is the one all prior SMAP results reference).

In [3]:
with open(f"{OUTPUT_PATH}/channel_data_normalized.pkl", "wb") as f:
    pickle.dump(channel_data_normalized, f)

config = {
    'window_size': WINDOW_SIZE,
    'stride_anomaly': STRIDE_ANOMALY,
    'stride_normal': STRIDE_NORMAL,
    'max_normal_windows': MAX_NORMAL,
    'normalization': 'MinMaxScaler per channel — fit on train only',
    'total_channels': len(channel_data_normalized),
}
with open(f"{OUTPUT_PATH}/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("saved channel_data_normalized.pkl and config.json to", OUTPUT_PATH)


saved channel_data_normalized.pkl and config.json to /kaggle/working
